In [2]:
import pandas as pd
import numpy as np

print("=== 1. DEFINITION DU PLAN D'AGENCEMENT INITIAL ===")

# Plan d'implantation physique initial des rayons dans le magasin (Allées 1 à 12)
plan_allees = {
    'Beurres & Fromages': 1, 'Crèmerie': 1,
    'Boucherie': 2, 'Poissonnerie': 2,
    'Fruits': 3, 'Légumes': 3,
    'Boulangerie': 4, 'Pâtisserie': 4,
    'Céréales & Confitures': 5, 'Café & Thé': 5,
    'Épicerie Sèche': 6, 
    'Huiles & Condiments': 7,
    'Sauces & Soupes': 8, 'Conserves': 8,
    'Snacks': 9, 'Confiserie': 9,
    'Sodas & Boissons': 10, 'Eaux & Jus': 10,
    'Surgelés': 11, 'Glaces': 11,
    'Hygiène & Soins': 12, 'Entretien Ménager': 12, 'Bébé': 12
}

# Chargement du fichier des résultats SIS recalculés
df_sis = pd.read_csv("score_incoherence_spatiale_sis_corrige.csv")

print("\n=== 2. GENERATION DES RECOMMANDATIONS D'AGENCEMENT ===")

# Filtrage des incohérences majeures (SIS > 1.50)
incoherences = df_sis[df_sis['Score_SIS'] > 1.50].drop_duplicates(subset=['Score_SIS']).copy()

print(f" {len(incoherences)} incohérence(s) majeure(s) détectée(s) nécessitant une réorganisation :\n")

recommandations = []

for idx, row in incoherences.iterrows():
    r1, r2 = row['Rayon_A'], row['Rayon_B']
    lift = row['Lift_Affinite']
    dist_actuelle = row['Distance_Physique_Allées']
    sis_actuel = row['Score_SIS']
    
    # Rapprochement cible : ramener la distance à 1 allée (adjacence)
    dist_cible = 1 if r1 != r2 else 0
    sis_cible = round(lift * dist_cible, 2)
    gain_sis = round(sis_actuel - sis_cible, 2)
    gain_pct = round((gain_sis / sis_actuel) * 100, 1)
    
    rec = {
        'Paire_Rayons': f"{r1} <---> {r2}",
        'Lift': lift,
        'Distance_Actuelle': f"{dist_actuelle} allée(s)",
        'SIS_Actuel': sis_actuel,
        'Distance_Optimisee': f"{dist_cible} allée",
        'SIS_Optimise': sis_cible,
        'Gain_SIS': f"-{gain_sis} (-{gain_pct}%)",
        'Action_Recommandee': f"Rapprocher '{r1}' et '{r2}' à 1 allée d'écart."
    }
    recommandations.append(rec)

df_reco = pd.DataFrame(recommandations)
display(df_reco[['Paire_Rayons', 'Lift', 'Distance_Actuelle', 'SIS_Actuel', 'SIS_Optimise', 'Gain_SIS']])


print("\n=== 3. SIMULATION DU NOUVEAU PLAN D'IMPLANTATION (PLAN CIBLE) ===")

# Copie et réallocation des allées pour le plan cible
plan_allees_optimise = plan_allees.copy()

# Ajustements ciblés pour réduire le SIS
plan_allees_optimise['Sauces & Soupes'] = 7  # Rapprochement de l'Épicerie Sèche (6) et des Huiles (7)
plan_allees_optimise['Conserves'] = 11        # Rapprochement du pôle alimentaire distant

print(" Réallocation des allées appliquée avec succès.")

# Calcul de l'impact sur le SIS moyen global
sis_global_actuel = df_sis['Score_SIS'].mean()

sis_simule = []
for idx, row in df_sis.iterrows():
    r1, r2 = row['Rayon_A'], row['Rayon_B']
    lift = row['Lift_Affinite']
    
    new_dist = abs(plan_allees_optimise[r1] - plan_allees_optimise[r2])
    new_sis = lift * new_dist
    sis_simule.append(new_sis)

sis_global_optimise = np.mean(sis_simule)
reduction_globale = round(((sis_global_actuel - sis_global_optimise) / sis_global_actuel) * 100, 1)

print("\n--- IMPACT GLOBAL DE L'OPTIMISATION ---")
print(f" • Score SIS Moyen Actuel   : {sis_global_actuel:.2f}")
print(f" • Score SIS Moyen Optimisé : {sis_global_optimise:.2f}")
print(f" • Réduction Globale des Incohérences Spatiales : -{reduction_globale}%")

=== 1. DEFINITION DU PLAN D'AGENCEMENT INITIAL ===

=== 2. GENERATION DES RECOMMANDATIONS D'AGENCEMENT ===
 2 incohérence(s) majeure(s) détectée(s) nécessitant une réorganisation :



,Paire_Rayons,Lift,Distance_Actuelle,SIS_Actuel,SIS_Optimise,Gain_SIS
0,Entretien Ménager <---> Conserves,1.00,4 allée(s),4.02,1.00,-3.02 (-75.1%)
1,Sauces & Soupes <---> Épicerie Sèche,1.04,2 allée(s),2.07,1.04,-1.03 (-49.8%)



=== 3. SIMULATION DU NOUVEAU PLAN D'IMPLANTATION (PLAN CIBLE) ===
 Réallocation des allées appliquée avec succès.

--- IMPACT GLOBAL DE L'OPTIMISATION ---
 • Score SIS Moyen Actuel   : 1.55
 • Score SIS Moyen Optimisé : 1.24
 • Réduction Globale des Incohérences Spatiales : -20.4%
